[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fvalenzuelag/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA1/IL1.2/3-chain-of-thought.ipynb)


# 3. Chain-of-Thought (CoT) Prompting - Razonamiento Paso a Paso

## Objetivos de Aprendizaje
- Comprender el concepto de Chain-of-Thought (CoT) prompting
- Implementar CoT con y sin ejemplos
- Aplicar CoT a problemas complejos y razonamiento lógico
- Combinar CoT con otras técnicas de prompting

## ¿Qué es Chain-of-Thought?

Chain-of-Thought (CoT) es una técnica que hace que el modelo "**piense en voz alta**" mostrando su proceso de razonamiento paso a paso antes de llegar a la respuesta final.

### Principio Básico:
En lugar de saltar directamente a la respuesta, el modelo:
1. **Descompone** el problema en pasos
2. **Razona** cada paso explícitamente  
3. **Construye** hacia la solución final
4. **Proporciona** la respuesta con justificación

### Ventajas:
- **Mejor precisión**: Especialmente en problemas complejos
- **Transparencia**: Puedes ver el razonamiento
- **Debugging**: Identificar dónde falla el razonamiento
- **Confianza**: Mayor seguridad en la respuesta

### Casos de Uso Ideales:
- Problemas matemáticos
- Razonamiento lógico
- Análisis complejos
- Toma de decisiones
- Resolución de problemas multi-paso

In [1]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q langchain langchain-classic langchain-openai python-dotenv


In [2]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
    os.environ.setdefault("LLM_MODEL_SMALL", "ministral-8b-latest")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

# Configuración inicial
from langchain_openai import ChatOpenAI
# LangChain v1: langchain_core.messages / langchain_core.documents
from langchain_classic.schema import HumanMessage
import os
import time

# Configurar el modelo
llm = ChatOpenAI(
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
    model=os.getenv("LLM_MODEL", "mistral-small-latest"),
    temperature=0.1  # Baja temperatura para razonamiento más consistente
)

print("✓ Modelo configurado para Chain-of-Thought")
print("✓ Temperature baja para razonamiento consistente")

✓ Modelo configurado para Chain-of-Thought
✓ Temperature baja para razonamiento consistente


## Comparación: Sin CoT vs Con CoT

Veamos la diferencia dramática que puede hacer CoT en problemas complejos.

In [3]:
# Comparación directa: razonamiento directo vs CoT
def comparar_sin_vs_con_cot():
    print("=== COMPARACIÓN: SIN CoT vs CON CoT ===")
    
    # Problema complejo que requiere múltiples pasos
    problema = """Una tienda tiene una promoción: 'Compra 2 productos y obtén 30% de descuento en el más barato'. 
    Juan compra una camiseta de 45€, unos zapatos de 120€ y una chaqueta de 80€. 
    ¿Cuánto paga en total?"""
    
    # Prompt sin CoT
    prompt_sin_cot = f"""Resuelve este problema:
    
{problema}
    
Respuesta:"""
    
    # Prompt con CoT
    prompt_con_cot = f"""Resuelve este problema paso a paso:
    
{problema}
    
Piensa paso a paso:
1. Primero identifica los productos y precios
2. Determina cómo se aplica la promoción
3. Calcula el descuento
4. Calcula el total final
    
Razonamiento:"""
    
    # Probar sin CoT
    print("\n1. SIN CHAIN-OF-THOUGHT:")
    print("-" * 30)
    try:
        response_sin = llm.invoke([HumanMessage(content=prompt_sin_cot)])
        print(response_sin.content)
    except Exception as e:
        print(f"Error: {e}")
    
    print("\n" + "="*60)
    
    # Probar con CoT
    print("\n2. CON CHAIN-OF-THOUGHT:")
    print("-" * 30)
    try:
        response_con = llm.invoke([HumanMessage(content=prompt_con_cot)])
        print(response_con.content)
    except Exception as e:
        print(f"Error: {e}")
    
    print("\n=== ANÁLISIS ===")
    print("• Sin CoT: Puede llegar a respuesta incorrecta o saltar pasos")
    print("• Con CoT: Muestra razonamiento completo y reduce errores")
    print("• CoT especialmente útil para problemas multi-paso")

# Ejecutar comparación
comparar_sin_vs_con_cot()

=== COMPARACIÓN: SIN CoT vs CON CoT ===

1. SIN CHAIN-OF-THOUGHT:
------------------------------


Para resolver el problema, seguiremos estos pasos:

1. **Identificar los productos y sus precios:**
   - Camiseta: 45€
   - Zapatos: 120€
   - Chaqueta: 80€

2. **Aplicar la promoción:**
   La promoción dice: *"Compra 2 productos y obtén 30% de descuento en el más barato"*.
   - Juan compra **3 productos**, por lo que puede aplicar la promoción **una vez** (ya que la promoción es por cada 2 productos).
   - Para maximizar el descuento, elegimos los **2 productos más caros** (zapatos y chaqueta) y aplicamos el 30% de descuento en el más barato de estos dos (chaqueta).

3. **Calcular el descuento:**
   - Precio de la chaqueta: 80€
   - Descuento: 30% de 80€ = 0.30 × 80 = 24€
   - Precio de la chaqueta con descuento: 80€ - 24€ = **56€**

4. **Sumar los precios finales:**
   - Camiseta: 45€ (sin descuento)
   - Zapatos: 120€ (sin descuento)
   - Chaqueta con descuento: 56€
   - **Total a pagar:** 45€ + 120€ + 56€ = **221€**

**Respuesta final:**
Juan paga **221€** en total.


2. CON CHAIN-

**Solución paso a paso:**

1. **Identifica los productos y sus precios:**
   - Camiseta: **45€**
   - Zapatos: **120€**
   - Chaqueta: **80€**

2. **Determina cómo se aplica la promoción:**
   - La promoción es: *"Compra 2 productos y obtén 30% de descuento en el más barato"*.
   - Juan compra **3 productos**, pero la promoción solo se aplica a **2 de ellos** (debe elegir cuáles dos).
   - Para maximizar el ahorro, Juan debe seleccionar los dos productos más caros (ya que el descuento se aplica al más barato de los dos).
   - **Productos seleccionados para la promoción:** Zapatos (120€) y Chaqueta (80€).
     - El más barato de estos dos es la **chaqueta (80€)**.

3. **Calcula el descuento:**
   - Descuento del 30% sobre la chaqueta:
     \( 80€ \times 0.30 = 24€ \).

4. **Calcula el total final:**
   - **Precio original de los productos:**
     \( 45€ (camiseta) + 120€ (zapatos) + 80€ (chaqueta) = 245€ \).
   - **Aplicar descuento:**
     \( 245€ - 24€ = 221€ \).

**Respuesta final:**

## Zero-Shot Chain-of-Thought

La forma más simple de CoT: simplemente pedirle al modelo que "piense paso a paso".

In [4]:
# Zero-shot CoT: solo agregar "piensa paso a paso"
def zero_shot_cot():
    print("=== ZERO-SHOT CHAIN-OF-THOUGHT ===")
    
    problemas = [
        "Si un tren viaja a 80 km/h y necesita llegar a una ciudad que está a 240 km, pero se detiene 15 minutos en una estación intermedia, ¿cuánto tiempo total toma el viaje?",
        "Una empresa tiene 150 empleados. El 40% trabaja en desarrollo, el 25% en ventas, el 20% en marketing y el resto en administración. Si cada empleado de desarrollo gana 50,000€ anuales, ¿cuál es el costo anual solo del departamento de desarrollo?",
        "María tiene el triple de edad que su hermana Ana. En 5 años, María tendrá el doble de la edad que tendrá Ana. ¿Cuántos años tiene cada una ahora?"
    ]
    
    for i, problema in enumerate(problemas, 1):
        print(f"\n{i}. PROBLEMA:")
        print(f"{problema}")
        
        # Prompt zero-shot CoT
        prompt = f"{problema}\n\nPiensa paso a paso:"
        
        try:
            response = llm.invoke([HumanMessage(content=prompt)])
            print("\nSOLUCIÓN:")
            print(response.content)
            
            # Análisis básico del razonamiento
            pasos = response.content.count('\n')
            tiene_calculo = any(op in response.content for op in ['+', '-', '*', '/', '=', '%'])
            
            print(f"\nAnálisis: {pasos} líneas de razonamiento, {'con' if tiene_calculo else 'sin'} cálculos explícitos")
            
        except Exception as e:
            print(f"Error: {e}")
        
        print("-" * 80)

# Ejecutar zero-shot CoT
zero_shot_cot()

=== ZERO-SHOT CHAIN-OF-THOUGHT ===

1. PROBLEMA:
Si un tren viaja a 80 km/h y necesita llegar a una ciudad que está a 240 km, pero se detiene 15 minutos en una estación intermedia, ¿cuánto tiempo total toma el viaje?



SOLUCIÓN:
Vamos a calcular el tiempo total del viaje paso a paso.

1. **Tiempo de viaje sin paradas:**

   - **Distancia:** 240 km
   - **Velocidad:** 80 km/h

   \[
   \text{Tiempo} = \frac{\text{Distancia}}{\text{Velocidad}} = \frac{240\,\text{km}}{80\,\text{km/h}} = 3\,\text{horas}
   \]

2. **Tiempo de parada en la estación:**

   - **Duración de la parada:** 15 minutos

   Convertimos los minutos a horas para mantener la coherencia:

   \[
   15\,\text{minutos} = \frac{15}{60}\,\text{horas} = 0.25\,\text{horas}
   \]

3. **Tiempo total del viaje:**

   \[
   \text{Tiempo total} = \text{Tiempo de viaje} + \text{Tiempo de parada} = 3\,\text{horas} + 0.25\,\text{horas} = 3.25\,\text{horas}
   \]

   Convertimos 0.25 horas a minutos:

   \[
   0.25\,\text{horas} \times 60\,\text{minutos/hora} = 15\,\text{minutos}
   \]

   Por lo tanto, el tiempo total es **3 horas y 15 minutos**.

**Respuesta final:**

\[
\boxed{3\,\text{horas y }15\,\text{minutos}}
\]

Análisis: 39 líneas de razona


SOLUCIÓN:
Vamos a resolver el problema paso a paso:

1. **Calcular el número de empleados en desarrollo:**
   - La empresa tiene 150 empleados.
   - El 40% trabaja en desarrollo.
   - Número de empleados en desarrollo = 40% de 150 = 0.40 × 150 = **60 empleados**.

2. **Calcular el costo anual del departamento de desarrollo:**
   - Cada empleado de desarrollo gana 50,000€ anuales.
   - Costo total = Número de empleados × Salario anual por empleado = 60 × 50,000€ = **3,000,000€**.

**Respuesta final:**
El costo anual solo del departamento de desarrollo es **3,000,000€**.

Análisis: 12 líneas de razonamiento, con cálculos explícitos
--------------------------------------------------------------------------------

3. PROBLEMA:
María tiene el triple de edad que su hermana Ana. En 5 años, María tendrá el doble de la edad que tendrá Ana. ¿Cuántos años tiene cada una ahora?



SOLUCIÓN:
Vamos a resolver el problema paso a paso.

**1. Definir las variables:**
- Sea \( A \) la edad actual de Ana.
- Como María tiene el triple de edad que Ana, la edad actual de María es \( 3A \).

**2. Plantear la ecuación para dentro de 5 años:**
- En 5 años, Ana tendrá \( A + 5 \) años.
- En 5 años, María tendrá \( 3A + 5 \) años.
- Según el problema, en 5 años María tendrá el doble de edad que Ana. Por lo tanto:
  \[
  3A + 5 = 2(A + 5)
  \]

**3. Resolver la ecuación:**
\[
\begin{align*}
3A + 5 &= 2(A + 5) \\
3A + 5 &= 2A + 10 \\
3A - 2A &= 10 - 5 \\
A &= 5
\end{align*}
\]

**4. Calcular la edad actual de María:**
\[
3A = 3 \times 5 = 15
\]

**5. Verificar la solución:**
- Edad actual de Ana: 5 años.
- Edad actual de María: 15 años.
- En 5 años:
  - Ana tendrá \( 5 + 5 = 10 \) años.
  - María tendrá \( 15 + 5 = 20 \) años.
- Comprobamos que \( 20 = 2 \times 10 \), lo cual es correcto.

**Respuesta final:**
- Ana tiene **5 años**.
- María tiene **15 años**.

Análisis: 39 lín

## Few-Shot Chain-of-Thought

Combinamos CoT con ejemplos para mostrar el patrón de razonamiento deseado.

In [5]:
# Few-shot CoT: ejemplos con razonamiento paso a paso
def few_shot_cot():
    print("=== FEW-SHOT CHAIN-OF-THOUGHT ===")
    
    # Nuevo problema para resolver
    nuevo_problema = "Un parking cobra 3€ la primera hora y 2€ cada hora adicional. Si alguien paga 15€, ¿cuántas horas estuvo estacionado?"
    
    # Prompt con ejemplos de razonamiento
    prompt_few_shot_cot = f"""Resuelve problemas matemáticos mostrando el razonamiento paso a paso:
    
Problema: Una pizza cuesta 12€ y cada ingrediente extra cuesta 1.50€. Si Pedro paga 18€, ¿cuántos ingredientes extra pidió?
Razonamiento:
1. Precio base de la pizza: 12€
2. Total pagado: 18€
3. Dinero gastado en extras: 18€ - 12€ = 6€
4. Costo por ingrediente extra: 1.50€
5. Número de ingredientes: 6€ ÷ 1.50€ = 4 ingredientes
Respuesta: Pedro pidió 4 ingredientes extra.
    
Problema: En una clase hay 24 estudiantes. Si se forman grupos de 6 estudiantes cada uno, ¿cuántos grupos se pueden formar? Si sobra algún estudiante, ¿cuántos?
Razonamiento:
1. Total de estudiantes: 24
2. Estudiantes por grupo: 6
3. División: 24 ÷ 6 = 4 grupos exactos
4. Verificación: 4 × 6 = 24 estudiantes
5. Resto: 24 - 24 = 0 estudiantes sobran
Respuesta: Se pueden formar 4 grupos completos y no sobra ningún estudiante.
    
Problema: {nuevo_problema}
Razonamiento:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_few_shot_cot)])
        print("PROBLEMA A RESOLVER:")
        print(nuevo_problema)
        print("\nSOLUCIÓN CON RAZONAMIENTO:")
        print(response.content)
        
        # Verificar si siguió el patrón
        razonamiento = response.content
        tiene_pasos_numerados = bool(re.search(r'\d+\.', razonamiento))
        tiene_calculos = any(op in razonamiento for op in ['=', '+', '-', '*', '/', '€'])
        tiene_respuesta_final = 'respuesta' in razonamiento.lower()
        
        print("\n=== ANÁLISIS DEL PATRÓN ===")
        print(f"✓ Pasos numerados: {'Sí' if tiene_pasos_numerados else 'No'}")
        print(f"✓ Cálculos explícitos: {'Sí' if tiene_calculos else 'No'}")
        print(f"✓ Respuesta final clara: {'Sí' if tiene_respuesta_final else 'No'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar few-shot CoT
import re
few_shot_cot()

=== FEW-SHOT CHAIN-OF-THOUGHT ===


PROBLEMA A RESOLVER:
Un parking cobra 3€ la primera hora y 2€ cada hora adicional. Si alguien paga 15€, ¿cuántas horas estuvo estacionado?

SOLUCIÓN CON RAZONAMIENTO:
**Problema:** Un parking cobra 3€ la primera hora y 2€ cada hora adicional. Si alguien paga 15€, ¿cuántas horas estuvo estacionado?

**Razonamiento:**
1. **Costo de la primera hora:** 3€ (incluye la primera hora).
2. **Dinero restante después de la primera hora:** 15€ - 3€ = 12€.
3. **Costo por hora adicional:** 2€.
4. **Número de horas adicionales:** 12€ ÷ 2€ = 6 horas.
5. **Total de horas:** 1 hora (base) + 6 horas (adicionales) = **7 horas**.

**Respuesta:** Estuvo estacionado **7 horas**.

=== ANÁLISIS DEL PATRÓN ===
✓ Pasos numerados: Sí
✓ Cálculos explícitos: Sí
✓ Respuesta final clara: Sí


## CoT para Razonamiento Lógico

Chain-of-Thought es especialmente poderoso para problemas de lógica y deducción.

In [6]:
# CoT para problemas de lógica
def cot_razonamiento_logico():
    print("=== CoT PARA RAZONAMIENTO LÓGICO ===")
    
    # Problema de lógica clásico
    problema_logica = """En una mesa redonda se sientan 5 personas: Ana, Bruno, Carlos, Diana y Elena.
    - Ana no está al lado de Bruno
    - Carlos está exactamente frente a Diana
    - Elena está al lado derecho de Ana
    - Bruno está al lado de Carlos
    
¿Cuál es la disposición completa alrededor de la mesa?"""
    
    prompt_logica = f"""Resuelve este problema de lógica paso a paso:
    
{problema_logica}
    
Razona sistemáticamente:
1. Identifica las restricciones
2. Establece relaciones conocidas
3. Deduce posiciones paso a paso
4. Verifica que se cumplan todas las condiciones
    
Proceso de deducción:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_logica)])
        print("PROBLEMA DE LÓGICA:")
        print(problema_logica)
        print("\nPROCESO DE RAZONAMIENTO:")
        print(response.content)
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar razonamiento lógico
cot_razonamiento_logico()

=== CoT PARA RAZONAMIENTO LÓGICO ===


PROBLEMA DE LÓGICA:
En una mesa redonda se sientan 5 personas: Ana, Bruno, Carlos, Diana y Elena.
    - Ana no está al lado de Bruno
    - Carlos está exactamente frente a Diana
    - Elena está al lado derecho de Ana
    - Bruno está al lado de Carlos

¿Cuál es la disposición completa alrededor de la mesa?

PROCESO DE RAZONAMIENTO:
Vamos a resolver el problema paso a paso, siguiendo el razonamiento sistemático que has propuesto.

---

### **1. Identifica las restricciones**
Tenemos 5 personas sentadas en una mesa redonda: **Ana (A), Bruno (B), Carlos (C), Diana (D) y Elena (E)**. Las restricciones son:
1. **Ana no está al lado de Bruno** (A no adyacente a B).
2. **Carlos está exactamente frente a Diana** (C frente a D).
3. **Elena está al lado derecho de Ana** (E a la derecha de A).
4. **Bruno está al lado de Carlos** (B adyacente a C).

---

### **2. Establece relaciones conocidas**
- **Restricción 2 (C frente a D):** En una mesa redonda con 5 personas, "frente" significa que hay **2

In [7]:
# CoT para análisis de casos complejos
def cot_analisis_complejo():
    print("=== CoT PARA ANÁLISIS COMPLEJO ===")
    
    # Caso de negocio complejo
    caso_negocio = """Una startup de software tiene las siguientes métricas:
    - 10,000 usuarios activos mensuales
    - Tasa de conversión a premium: 5%
    - Precio premium: 29€/mes
    - Costo de adquisición por usuario: 15€
    - Retención mensual: 85%
    - Costos operativos mensuales: 12,000€
    
La empresa está considerando reducir el precio a 19€/mes para aumentar la conversión a 8%. 
¿Es una buena decisión financiera?"""
    
    prompt_analisis = f"""Analiza este caso de negocio paso a paso:
    
{caso_negocio}
    
Estructura tu análisis:
1. Calcula métricas del escenario actual
2. Calcula métricas del escenario propuesto
3. Compara ingresos y costos
4. Considera factores adicionales
5. Proporciona recomendación fundamentada
    
Análisis detallado:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_analisis)])
        print("CASO DE NEGOCIO:")
        print(caso_negocio)
        print("\nANÁLISIS PASO A PASO:")
        print(response.content)
        
        # Verificar completitud del análisis
        analisis = response.content.lower()
        elementos_clave = [
            'ingresos', 'costos', 'beneficio', 'actual', 'propuesto', 
            'recomendación', 'conversión', 'usuarios'
        ]
        elementos_presentes = sum(1 for elemento in elementos_clave if elemento in analisis)
        
        print(f"\n=== COMPLETITUD DEL ANÁLISIS ===")
        print(f"✓ Elementos clave cubiertos: {elementos_presentes}/{len(elementos_clave)}")
        print(f"✓ Análisis completo: {'Sí' if elementos_presentes >= 6 else 'Parcial'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar análisis complejo
cot_analisis_complejo()

=== CoT PARA ANÁLISIS COMPLEJO ===


CASO DE NEGOCIO:
Una startup de software tiene las siguientes métricas:
    - 10,000 usuarios activos mensuales
    - Tasa de conversión a premium: 5%
    - Precio premium: 29€/mes
    - Costo de adquisición por usuario: 15€
    - Retención mensual: 85%
    - Costos operativos mensuales: 12,000€

La empresa está considerando reducir el precio a 19€/mes para aumentar la conversión a 8%. 
¿Es una buena decisión financiera?

ANÁLISIS PASO A PASO:
### **Análisis del Caso de Negocio: Reducción de Precio Premium**

#### **1. Métricas del Escenario Actual**
- **Usuarios activos mensuales (UAM):** 10,000
- **Tasa de conversión a premium:** 5% → **Usuarios premium:** 500 (10,000 × 0.05)
- **Ingresos mensuales por premium:** 500 × 29€ = **14,500€**
- **Costo de adquisición por usuario (CAC):** 15€ → **Costo total de adquisición:** 10,000 × 15€ = **150,000€**
- **Retención mensual:** 85% → **Usuarios retenidos al mes siguiente:** 8,500 (10,000 × 0.85)
- **Costos operativos mensuales:** 12,000€

*

## Técnicas Avanzadas de CoT

In [8]:
# Técnica 1: CoT con Auto-Verificación
def cot_con_verificacion():
    print("=== CoT CON AUTO-VERIFICACIÓN ===")
    
    problema = "Una piscina se llena con dos bombas. La bomba A la llena en 4 horas, la bomba B en 6 horas. Si funcionan juntas, ¿en cuánto tiempo llenan la piscina?"
    
    prompt_verificacion = f"""Resuelve este problema paso a paso y luego verifica tu respuesta:
    
{problema}
    
PASO 1 - RESOLUCIÓN:
Piensa paso a paso para encontrar la solución.
    
PASO 2 - VERIFICACIÓN:
Revisa tu cálculo usando un método diferente o verificando que los números tienen sentido.
    
PASO 3 - RESPUESTA FINAL:
Confirma tu respuesta final.
    
Proceso completo:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_verificacion)])
        print("PROBLEMA:")
        print(problema)
        print("\nSOLUCIÓN CON AUTO-VERIFICACIÓN:")
        print(response.content)
        
        # Verificar estructura
        contenido = response.content.lower()
        tiene_resolucion = 'paso 1' in contenido or 'resolución' in contenido
        tiene_verificacion = 'paso 2' in contenido or 'verificación' in contenido
        tiene_final = 'paso 3' in contenido or 'final' in contenido
        
        print("\n=== ESTRUCTURA ===")
        print(f"✓ Resolución: {'Sí' if tiene_resolucion else 'No'}")
        print(f"✓ Verificación: {'Sí' if tiene_verificacion else 'No'}")
        print(f"✓ Respuesta final: {'Sí' if tiene_final else 'No'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar CoT con verificación
cot_con_verificacion()

=== CoT CON AUTO-VERIFICACIÓN ===


PROBLEMA:
Una piscina se llena con dos bombas. La bomba A la llena en 4 horas, la bomba B en 6 horas. Si funcionan juntas, ¿en cuánto tiempo llenan la piscina?

SOLUCIÓN CON AUTO-VERIFICACIÓN:
**PASO 1 - RESOLUCIÓN:**

Para resolver este problema, seguiremos estos pasos:

1. **Determinar la tasa de llenado de cada bomba:**
   - La bomba A llena la piscina en 4 horas, por lo que su tasa de llenado es \( \frac{1}{4} \) de la piscina por hora.
   - La bomba B llena la piscina en 6 horas, por lo que su tasa de llenado es \( \frac{1}{6} \) de la piscina por hora.

2. **Calcular la tasa combinada de las dos bombas:**
   - Cuando trabajan juntas, sus tasas se suman:
     \[
     \text{Tasa combinada} = \frac{1}{4} + \frac{1}{6}
     \]
   - Para sumar las fracciones, encontramos un denominador común (12):
     \[
     \frac{1}{4} = \frac{3}{12}, \quad \frac{1}{6} = \frac{2}{12}
     \]
     \[
     \text{Tasa combinada} = \frac{3}{12} + \frac{2}{12} = \frac{5}{12} \text{ de la piscina por hor

In [9]:
# Técnica 2: CoT Multi-Perspectiva
def cot_multi_perspectiva():
    print("=== CoT MULTI-PERSPECTIVA ===")
    
    dilema = """Una empresa de delivery está considerando implementar un algoritmo de IA 
    para optimizar rutas que podría reducir costos en 20% pero eliminaría 100 empleos 
    de repartidores. ¿Debería implementarlo?"""
    
    prompt_multi = f"""Analiza este dilema desde múltiples perspectivas:
    
{dilema}
    
Analiza paso a paso desde cada perspectiva:
    
PERSPECTIVA 1 - FINANCIERA:
- Beneficios económicos
- Costos de implementación
- ROI a corto y largo plazo
    
PERSPECTIVA 2 - SOCIAL/ÉTICA:
- Impacto en empleados
- Responsabilidad social corporativa
- Percepción pública
    
PERSPECTIVA 3 - ESTRATÉGICA:
- Competitividad del mercado
- Innovación tecnológica
- Sostenibilidad del negocio
    
SÍNTESIS:
- Integra las perspectivas
- Propone soluciones alternativas
- Recomienda curso de acción
    
Análisis completo:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_multi)])
        print("DILEMA EMPRESARIAL:")
        print(dilema)
        print("\nANÁLISIS MULTI-PERSPECTIVA:")
        print(response.content)
        
        # Verificar cobertura de perspectivas
        analisis = response.content.lower()
        perspectivas = ['financiera', 'social', 'ética', 'estratégica', 'síntesis']
        perspectivas_cubiertas = sum(1 for p in perspectivas if p in analisis)
        
        print(f"\n=== COBERTURA DEL ANÁLISIS ===")
        print(f"✓ Perspectivas cubiertas: {perspectivas_cubiertas}/{len(perspectivas)}")
        print(f"✓ Análisis integral: {'Sí' if perspectivas_cubiertas >= 4 else 'Parcial'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar análisis multi-perspectiva
cot_multi_perspectiva()

=== CoT MULTI-PERSPECTIVA ===


DILEMA EMPRESARIAL:
Una empresa de delivery está considerando implementar un algoritmo de IA 
    para optimizar rutas que podría reducir costos en 20% pero eliminaría 100 empleos 
    de repartidores. ¿Debería implementarlo?

ANÁLISIS MULTI-PERSPECTIVA:
### **Análisis Multiperspectivo de la Implementación del Algoritmo de IA en la Empresa de Delivery**

---

## **1. PERSPECTIVA FINANCIERA**

### **Beneficios económicos**
- **Reducción de costos operativos**: Un ahorro del **20%** en rutas implica una mejora significativa en la eficiencia. Si la empresa maneja un volumen alto de entregas (ej. 10,000 pedidos/día), esto podría traducirse en **millones de dólares anuales** en ahorros.
- **Mayor capacidad de escalabilidad**: Con rutas optimizadas, la empresa podría manejar más pedidos sin aumentar proporcionalmente la flota de repartidores, reduciendo costos fijos (vehículos, seguros, mantenimiento).
- **Ventaja competitiva**: Menores costos podrían permitir precios más bajos o márgenes má

## CoT para Debugging de Código

Una aplicación práctica muy útil: usar CoT para analizar y debuggear código.

In [10]:
# CoT para debugging de código
def cot_debugging():
    print("=== CoT PARA DEBUGGING DE CÓDIGO ===")
    
    codigo_con_bug = '''def calcular_promedio(numeros):
    total = 0
    for numero in numeros:
        total += numero
    promedio = total / len(numeros)
    return promedio

# Uso
datos = []
resultado = calcular_promedio(datos)
print(f"El promedio es: {resultado}")'''
    
    prompt_debugging = f"""Analiza este código paso a paso para encontrar problemas:
    
```python
{codigo_con_bug}
```
    
PASO 1 - COMPRENSIÓN:
¿Qué se supone que hace este código?
    
PASO 2 - ANÁLISIS LÍNEA POR LÍNEA:
Examina cada línea en busca de problemas potenciales.
    
PASO 3 - IDENTIFICACIÓN DE PROBLEMAS:
¿Qué errores o problemas específicos encuentras?
    
PASO 4 - CASOS PROBLEMÁTICOS:
¿En qué situaciones fallaría este código?
    
PASO 5 - SOLUCIÓN:
¿Cómo arreglarías estos problemas?
    
Análisis de debugging:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_debugging)])
        print("CÓDIGO A ANALIZAR:")
        print(codigo_con_bug)
        print("\nANÁLISIS DE DEBUGGING:")
        print(response.content)
        
        # Verificar si encontró el problema principal
        analisis = response.content.lower()
        encontro_division_cero = any(term in analisis for term in ['división', 'cero', 'vacía', 'empty'])
        propuso_solucion = 'if' in analisis or 'len(' in analisis or 'excepción' in analisis
        
        print(f"\n=== EFECTIVIDAD DEL DEBUGGING ===")
        print(f"✓ Identificó división por cero: {'Sí' if encontro_division_cero else 'No'}")
        print(f"✓ Propuso solución: {'Sí' if propuso_solucion else 'No'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar debugging
cot_debugging()

=== CoT PARA DEBUGGING DE CÓDIGO ===


CÓDIGO A ANALIZAR:
def calcular_promedio(numeros):
    total = 0
    for numero in numeros:
        total += numero
    promedio = total / len(numeros)
    return promedio

# Uso
datos = []
resultado = calcular_promedio(datos)
print(f"El promedio es: {resultado}")

ANÁLISIS DE DEBUGGING:
### **PASO 1 - COMPRENSIÓN**
El código define una función `calcular_promedio(numeros)` que:
1. Inicializa una variable `total` en 0.
2. Itera sobre una lista `numeros`, sumando cada elemento a `total`.
3. Calcula el promedio dividiendo `total` entre la cantidad de elementos (`len(numeros)`).
4. Retorna el promedio.

Luego, se usa la función con una lista vacía `datos = []` y se imprime el resultado.

---

### **PASO 2 - ANÁLISIS LÍNEA POR LÍNEA**
1. **Definición de la función**:
   ```python
   def calcular_promedio(numeros):
   ```
   - No hay problema aquí, pero no se especifica el tipo de `numeros` (debería ser una lista o iterable de números).

2. **Inicialización de `total`**:
   ```python
   tota

# Ejercicio con Chain of Thought

In [11]:
# Ejercicio final: Diseña tu propio CoT prompt
def ejercicio_cot():
    print("=== EJERCICIO: DISEÑA TU CoT PROMPT ===")
    print("\nTarea: Crear un sistema CoT para análisis de inversión")
    print("\nEscenario:")
    print("Una persona tiene 10,000€ para invertir y está considerando tres opciones:")
    print("1. Acciones de tech (retorno esperado 12% anual, riesgo alto)")
    print("2. Bonos gubernamentales (retorno 3% anual, riesgo bajo)")
    print("3. Fondo mixto (retorno 7% anual, riesgo medio)")
    print("\nLa persona es joven (25 años) y puede asumir riesgo moderado.")
    
    # Template para el estudiante
    template_cot = """
    # TU PROMPT CoT AQUÍ:
    
    Analiza esta decisión de inversión paso a paso:
    
    [ESCENARIO]
    
    PASO 1 - PERFIL DEL INVERSOR:
    - Analiza edad, tolerancia al riesgo, horizonte temporal
    
    PASO 2 - ANÁLISIS DE OPCIONES:
    - Evalúa cada opción: retorno, riesgo, liquidez
    
    PASO 3 - ESTRATEGIA DE DIVERSIFICACIÓN:
    - Considera combinar opciones
    
    PASO 4 - RECOMENDACIÓN:
    - Proporciona recomendación específica con justificación
    
    Análisis de inversión:
    """
    
    print("\nDiseña un prompt CoT estructurado:")
    print(template_cot)
    
    # Prompt de ejemplo bien diseñado
    prompt_ejemplo = """Analiza esta decisión de inversión usando razonamiento paso a paso:
    
SITUACIÓN:
Inversor de 25 años con 10,000€ considerando:
- Acciones tech: 12% retorno anual, riesgo alto
- Bonos: 3% retorno anual, riesgo bajo  
- Fondo mixto: 7% retorno anual, riesgo medio
Tolerancia: riesgo moderado
    
PASO 1 - PERFIL DEL INVERSOR:
Analiza edad, horizonte temporal y tolerancia al riesgo.
    
PASO 2 - EVALUACIÓN DE OPCIONES:
Para cada opción, calcula:
- Valor esperado en 10 años
- Nivel de riesgo vs. perfil
- Pros y contras específicos
    
PASO 3 - ESTRATEGIA DE PORTFOLIO:
Considera distribución óptima entre opciones basada en:
- Diversificación de riesgo
- Maximización de retorno ajustado por riesgo
- Liquidez y flexibilidad
    
PASO 4 - RECOMENDACIÓN FINAL:
Proporciona distribución específica (porcentajes) con:
- Justificación detallada
- Proyección a 10 años
- Consideraciones adicionales
    
Análisis completo:"""
    
    print("\n=== PROMPT DE REFERENCIA ===")
    print(prompt_ejemplo)
    
    # Ejecutar el prompt ejemplo
    print("\n=== RESULTADO DEL ANÁLISIS ===")
    try:
        response = llm.invoke([HumanMessage(content=prompt_ejemplo)])
        print(response.content)
        
        # Análisis de la respuesta
        analisis = response.content.lower()
        elementos = ['perfil', 'evaluación', 'portfolio', 'recomendación', '%', '€', 'años']
        elementos_presentes = sum(1 for elem in elementos if elem in analisis)
        
        print(f"\n=== CALIDAD DEL ANÁLISIS ===")
        print(f"✓ Elementos clave: {elementos_presentes}/{len(elementos)}")
        print(f"✓ Análisis completo: {'Sí' if elementos_presentes >= 5 else 'Parcial'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar ejercicio
ejercicio_cot()

=== EJERCICIO: DISEÑA TU CoT PROMPT ===

Tarea: Crear un sistema CoT para análisis de inversión

Escenario:
Una persona tiene 10,000€ para invertir y está considerando tres opciones:
1. Acciones de tech (retorno esperado 12% anual, riesgo alto)
2. Bonos gubernamentales (retorno 3% anual, riesgo bajo)
3. Fondo mixto (retorno 7% anual, riesgo medio)

La persona es joven (25 años) y puede asumir riesgo moderado.

Diseña un prompt CoT estructurado:

    # TU PROMPT CoT AQUÍ:

    Analiza esta decisión de inversión paso a paso:

    [ESCENARIO]

    PASO 1 - PERFIL DEL INVERSOR:
    - Analiza edad, tolerancia al riesgo, horizonte temporal

    PASO 2 - ANÁLISIS DE OPCIONES:
    - Evalúa cada opción: retorno, riesgo, liquidez

    PASO 3 - ESTRATEGIA DE DIVERSIFICACIÓN:
    - Considera combinar opciones

    PASO 4 - RECOMENDACIÓN:
    - Proporciona recomendación específica con justificación

    Análisis de inversión:
    

=== PROMPT DE REFERENCIA ===
Analiza esta decisión de inversión usa

### **Análisis de Inversión: 10,000€ para un Inversor de 25 Años con Tolerancia al Riesgo Moderado**

---

## **PASO 1: PERFIL DEL INVERSOR**
### **1.1. Edad y Horizonte Temporal**
- **Edad:** 25 años.
- **Horizonte temporal:** Largo plazo (10+ años).
  - **Implicaciones:**
    - Mayor capacidad para asumir riesgo debido a la juventud (tiempo para recuperarse de caídas del mercado).
    - Flexibilidad para invertir en activos con mayor volatilidad (ej. acciones) sin necesidad de liquidez inmediata.
    - Posibilidad de aprovechar el **efecto del interés compuesto** durante décadas.

### **1.2. Tolerancia al Riesgo**
- **Moderada:** Prefiere un equilibrio entre crecimiento y estabilidad.
  - **Señales:**
    - No quiere perder más del 20-30% en un año (ej. caída del mercado).
    - Busca retornos superiores a los bonos pero sin la volatilidad extrema de las acciones individuales.

### **1.3. Objetivos Financieros**
- **Crecimiento del capital** (no solo preservación).
- **Diversificació

## Limitaciones y Consideraciones de CoT

### ✅ Cuándo Usar CoT:
- Problemas matemáticos complejos
- Razonamiento lógico multi-paso
- Análisis que requiere transparencia
- Cuando necesitas verificar el proceso
- Problemas donde el "por qué" es importante

### ⚠️ Limitaciones:
- **Más tokens**: Respuestas más largas = mayor costo
- **Tiempo**: Razonamiento paso a paso toma más tiempo
- **Complejidad innecesaria**: Para problemas simples puede ser excesivo
- **Razonamiento erróneo**: Puede mostrar lógica incorrecta convincente

### 🎯 Mejores Prácticas:
1. **Estructura clara**: Define pasos específicos
2. **Verificación**: Incluye auto-verificación cuando sea posible
3. **Ejemplos**: Usa few-shot para mostrar patrón deseado
4. **Temperatura baja**: Para razonamiento más consistente
5. **Validación externa**: Verifica respuestas críticas independientemente

## Conceptos Clave Aprendidos

1. **Chain-of-Thought** mejora dramáticamente la precisión en problemas complejos
2. **Zero-shot CoT** es tan simple como agregar "piensa paso a paso"
3. **Few-shot CoT** combina ejemplos con razonamiento para mejor control
4. **Estructura explícita** guía el razonamiento hacia análisis completos
5. **Auto-verificación** aumenta la confiabilidad de las respuestas

## Próximos Pasos

En el siguiente notebook exploraremos **Técnicas Avanzadas** como Tree of Thoughts (ToT), Self-Consistency, y otras metodologías cutting-edge que llevan el prompt engineering al siguiente nivel.

### Para Practicar:
1. Aplica CoT a problemas de tu dominio específico
2. Experimenta con diferentes niveles de estructura
3. Compara precisión con y sin CoT
4. Desarrolla patrones de verificación personalizados